# Μάθημα 01 - Εισαγωγή στους Πράκτορες Τεχνητής Νοημοσύνης

Καλώς ήρθατε στο πρώτο μάθημα του μαθήματος **Πράκτορες Τεχνητής Νοημοσύνης για Αρχάριους**!

Ένας **πράκτορας ΤΝ** είναι ένα πρόγραμμα που χρησιμοποιεί ένα μεγάλο γλωσσικό μοντέλο (LLM) ως μηχανισμό συλλογισμού και μπορεί να παίρνει *ενέργειες* στον πραγματικό κόσμο — καλώντας APIs, ρωτώντας βάσεις δεδομένων ή εκτελώντας κώδικα — για να επιτύχει έναν στόχο εκ μέρους του χρήστη.

Σε αυτό το notebook θα κατασκευάσετε τον πρώτο σας πράκτορα: έναν **Πράκτορα Ταξιδιών** που προτείνει προορισμούς διακοπών. Στην πορεία θα μάθετε πώς να:

1. Συνδεθείτε στην Υπηρεσία Πράκτορα Microsoft Foundry χρησιμοποιώντας το **Πλαίσιο Πράκτορα Microsoft**.
2. Δώσετε στον πράκτορα ένα **εργαλείο** — μια απλή συνάρτηση Python που μπορεί να καλέσει.
3. Εκτελέσετε τον πράκτορα και να εξετάσετε την απόκρισή του.
4. Μεταδώσετε την απόκριση του πράκτορα διακριτό-τοκεν-διακριτό.


## Ρύθμιση

Πριν εκτελέσετε αυτό το notebook, βεβαιωθείτε ότι έχετε:

1. **Ένα έργο Microsoft Foundry** με αναπτυγμένο μοντέλο συνομιλίας (π.χ. `gpt-5-mini`).
2. **Έχετε συνδεθεί με το Azure CLI** — τρέξτε `az login` στο τερματικό σας.
3. **Ορίστε τις απαιτούμενες μεταβλητές περιβάλλοντος:**
   - `AZURE_AI_PROJECT_ENDPOINT` — το endpoint του έργου σας στο Microsoft Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — το όνομα του αναπτυγμένου μοντέλου σας.

Το παρακάτω κελί εγκαθιστά τα πακέτα Python που χρειάζεστε.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Δημιουργώντας τον Πρώτο σας Πράκτορα

Ένας πράκτορας χρειάζεται δύο πράγματα:

- **Οδηγίες** που του λένε *ποιος είναι* και *πώς να συμπεριφέρεται* (ένα σύστημα προτροπής).
- **Εργαλεία** — συναρτήσεις Python διακοσμημένες με `@tool` που ο πράκτορας μπορεί να καλέσει για να αντλήσει πληροφορίες ή να εκτελέσει ενέργειες.

Παρακάτω ορίζουμε ένα απλό εργαλείο που επιστρέφει μια λίστα με δημοφιλείς προορισμούς διακοπών. Ο πράκτορας θα χρησιμοποιήσει αυτό το εργαλείο όταν ένας χρήστης ζητήσει προτάσεις ταξιδιών.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Ροή Απαντήσεων

Για μια πιο διαδραστική εμπειρία, μπορείτε να **ροή** της απάντησης του πράκτορα. Αντί να περιμένετε για την πλήρη απάντηση, ο πράκτορας παραδίδει τμήματα κειμένου καθώς παράγονται. Αυτό είναι ιδιαίτερα χρήσιμο σε διεπαφές συζητήσεων όπου θέλετε να εμφανίσετε την έξοδο σε πραγματικό χρόνο.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Περίληψη

Σε αυτό το μάθημα μάθατε πώς να:

- **Δημιουργήσετε έναν πάροχο** που συνδέεται με την υπηρεσία Microsoft Foundry Agent μέσω του `FoundryChatClient`.
- **Ορίσετε ένα εργαλείο** χρησιμοποιώντας το διακοσμητή `@tool` έτσι ώστε ο πράκτορας να μπορεί να καλέσει τις συναρτήσεις Python σας.
- **Τρέξετε τον πράκτορα** με ένα μήνυμα χρήστη και να εκτυπώσετε την απάντησή του.
- **Ροή απαντήσεων** για έξοδο σε πραγματικό χρόνο.

Στο επόμενο μάθημα θα εξερευνήσουμε τα πλαίσια agentic πιο σε βάθος και θα μάθουμε πώς να δώσουμε στους πράκτορες πιο ισχυρά εργαλεία και δυνατότητες πολυβηματικής λογικής.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
